# MOVIES.CSV 

Este notebook documenta el análisis exploratorio y limpieza del conjunto de datos movies.csv


**Entrada**: movies.csv \
**Objetivos**: lectura, validación, limpieza y transformación \
**Salida**: movies.parquet


## Descripción del proceso

**Análisis y comprensión**
- movieId: identificador entero y único. CLAVE
- title: título de la película y año entre paréntesis
- genres: géneros separados por | 

**Validación**
- Tipos de datos
- Valores nulos
- MovieId único
- Title con año incluido
- Genres válidos 

**Limpieza**
- Valores nulos solo permitidos en year y genres cambio a NULL
- En valores duplicados de movieId se conserva uno
- Title sin año, sin espacios en blanco, sin caracteres extraños 
- películas sin género se conservan para filtrado colaborativo

**Transformación**
- Separación de columna title en title y year.
- Salida archivo .parquet




In [1]:
import pandas as pd

## Análisis y comprensión del dataset 

In [2]:
# Lectura de datos
movies = pd.read_csv("../data/01_raw/movielens/movies.csv")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
movies.size

29226

In [4]:
movies.dtypes

movieId    int64
title        str
genres       str
dtype: object

In [5]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB


Tipos correctos y sin valores null

## Validación

In [6]:
# movieId entero
(movies['movieId'] % 1 == 0).sum() == movies['movieId'].size

np.True_

In [7]:
# movieId único
movies['movieId'].nunique() == movies['movieId'].size

True

Lista de géneros permitidos, obtenidos de la documentación de movieLens.

In [8]:
genresVal = ["Action", "Adventure", "Animation", "Children", "Comedy", "Crime", "Documentary", "Drama", 
"Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western", "IMAX",
"(no genres listed)"]

In [9]:
# crear uana columna con listas de los géneros 
genresList = movies['genres'].str.split("|")
print(genresList)

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9737                 [Action, Animation, Comedy, Fantasy]
9738                         [Animation, Comedy, Fantasy]
9739                                              [Drama]
9740                                  [Action, Animation]
9741                                             [Comedy]
Name: genres, Length: 9742, dtype: object


In [10]:
# comprobación que los géneros pertenecen a la lista de géneros válidos.
#apply aplica a cada elemento de la columna
validos = genresList.apply(lambda lista: all(g in genresVal for g in lista))

validos.sum() == movies['movieId'].size


np.True_

In [11]:
#verificar si las películas aparecen con el año
movies['title'].str.contains(r"\([0-9]{4}\)")
#peliculas sin año
len(movies[~movies['title'].str.contains(r"\([0-9]{4}\)")])

#Existen películas sin año

13

Si hubiera movieId duplicado se conservará solo un registro. 

In [12]:
movies = movies.drop_duplicates(subset="movieId", keep="first")

## Limpieza

In [13]:
# para genres (no genres listed) cambio a na
movies['genres'] = movies['genres'].replace("(no genres listed)", pd.NA)

## TRANSFORMACIÓN

In [14]:
# separar el año en una columna nueva, NA para año desconocido
movies["year"] = movies["title"].str.extract(r"\(([0-9]{4})\)").astype("Int64")
movies["title"] = movies["title"].str.replace(r"\([0-9]{4}\)$", "", regex=True)
movies

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,Action|Animation|Comedy|Fantasy,2017
9738,193583,No Game No Life: Zero,Animation|Comedy|Fantasy,2017
9739,193585,Flint,Drama,2017
9740,193587,Bungo Stray Dogs: Dead Apple,Action|Animation,2018


In [16]:
movies.to_parquet('../data/02_processed/movies_clean.parquet', index=False)